# Modèle QUATRE : français + kirundi (Google Colab)

On **continue la lecture du modèle trois** (même tokenizer). Pas de nouveau kirundi : on corrige ce que le run trois a montré.

| # | Correction par rapport au modèle trois |
|---|---|
| 1 | Bible **24 000** passages dans `dataset_quatre` (trois : 9 000, vieux dataset sur Kaggle), x2 |
| 2 | **Contrôle des fichiers** au départ (`attendu.json`) : arrêt si le dataset Kaggle n'est pas à jour |
| 3 | **Évaluation honnête** : mise de côté AVANT les répétitions, même mesure pendant et à la fin |
| 4 | Lecture **2 époques** (~2 h au lieu de 7 h), Wikipédia FR réduit (12 000 paragraphes) |
| 5 | Répétitions baissées (x50/x40/x30 -> x10), plus de traduction (FLORES dev x6, fr_rn x4), 6 époques |
| 6 | Fin de phrase gardée quand un exemple est coupé ; plus de réponse vide aux tests |
| 7 | Comparaison trois / quatre **sur tous les tests** (l'historique compte) |

| Étape | Ce qui se passe |
|---|---|
| 0 | Préparer : trouver les dossiers, l'heure de fin de session |
| 1 | Données publiques (Wikipédia FR 001 à 004, ~20 min) |
| 2 | **Lecture** (~2 h) |
| 3 | **Discussion** (~2,5 h) |
| 4 | Tests : modèle trois et modèle quatre, puis comparaison |
| 5 | Essayer le modèle, puis le `.zip` |

**Sauvegardes** : petite sauvegarde tous les 50 pas, copie complète toutes les 4 h,
et **pause propre avant la fin de la session**. Si ce n'est pas fini, la **version suivante** reprend au même pas.

**Avant de lancer** : dans **Mon Drive** (à la racine) il faut :
1. `dataset/`, `dataset_un/`, `dataset_deux/`, `dataset_trois/` et `dataset_quatre/` (les dossiers `datasets/...` de la machine)
2. `modele_trois/lecture/finale/` (point de départ) et `modele_trois/chat/finale/` (pour comparer les tests).
   Sur Kaggle : onglet Output du notebook trois_kaggle > télécharger, puis le mettre dans Drive.

Menu **Exécution > Modifier le type d'exécution > GPU T4**, puis **Exécuter tout**.
**Si Colab coupe** : relance tout (Exécuter tout). Chaque étape reprend au dernier point sauvegardé dans Drive.

## Étape 0 : préparer Colab

In [ ]:
import glob, os, shutil, time
DEBUT = time.time()
HEURES_SESSION = 0         # 0 = pas de limite connue : Colab peut couper, on reprend grâce aux sauvegardes
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DATASET = "/content/drive/MyDrive/dataset"
DATASET_UN = "/content/drive/MyDrive/dataset_un"
DATASET_DEUX = "/content/drive/MyDrive/dataset_deux"
DATASET_TROIS = "/content/drive/MyDrive/dataset_trois"
DATASET_QUATRE = "/content/drive/MyDrive/dataset_quatre"
TROIS = "/content/drive/MyDrive/modele_trois"
TRAVAIL = "/content/drive/MyDrive/modele_quatre"    # écriture dans Drive (survit aux coupures)
for d, quoi in [(f"{DATASET}/dialogues", "dataset"), (f"{DATASET_UN}/insiguro", "dataset_un"),
                (f"{DATASET_DEUX}/ikibonezamvugo", "dataset_deux"), (f"{DATASET_TROIS}/lecture_rn", "dataset_trois"),
                (f"{DATASET_QUATRE}/traduction", "dataset_quatre"),
                (f"{TROIS}/lecture/finale/config.json", "modele_trois/lecture/finale")]:
    if not os.path.exists(d):
        raise SystemExit(f"Manque dans Mon Drive : {quoi} (voir en haut).")
CODE = "/content/code_quatre"
shutil.copytree(f"{DATASET_QUATRE}/code", CODE, dirs_exist_ok=True)
!pip install -q -r {CODE}/requirements.txt
PUBLICS = "/content/drive/MyDrive/publics"

In [ ]:
# 0.b  Commun : variables lues par train.py, heure de fin, état de chaque étape
import torch
os.makedirs(f"{TRAVAIL}/journaux", exist_ok=True)
J = f"{TRAVAIL}/journaux"
os.environ.update(DATASET=DATASET, DATASET_UN=DATASET_UN, DATASET_DEUX=DATASET_DEUX, DATASET_TROIS=DATASET_TROIS,
                  DATASET_QUATRE=DATASET_QUATRE, TROIS=TROIS, TRAVAIL=TRAVAIL,
                  PUBLICS=PUBLICS)
if HEURES_SESSION:
    os.environ["FIN_SESSION"] = str(DEBUT + HEURES_SESSION * 3600)
    print("Pause prévue vers", time.strftime("%H:%M", time.localtime(DEBUT + HEURES_SESSION * 3600 - 25 * 60)))
NB_GPU = max(1, torch.cuda.device_count())
LANCER = f"torchrun --nproc_per_node {NB_GPU} train.py" if NB_GPU > 1 else "python -u train.py"
print(f"{NB_GPU} GPU -> {LANCER}")
os.chdir(CODE)
PAUSE = False

def fait(chemin):
    return os.path.exists(chemin)

def apres(etape, dossier):
    """Après une étape : finie, en pause (reprise à la version suivante), ou en échec."""
    global PAUSE
    if fait(f"{dossier}/finale/config.json"):
        print(f"OK : {etape} terminée -> {dossier}/finale")
    elif fait(f"{dossier}/PAUSE"):
        PAUSE = True
        print(f"PAUSE : {etape} pas finie (temps de session). Les étapes suivantes attendent la version suivante.")
    else:
        raise SystemExit(f"ÉCHEC : {etape}. Regarde {J}/{etape}.txt")

def meilleur_modele():
    for ph in ("chat", "lecture"):
        for sous in ("finale", "meilleure"):
            if fait(f"{TRAVAIL}/{ph}/{sous}/config.json"):
                return f"{TRAVAIL}/{ph}/{sous}"
    return f"{TROIS}/lecture/finale"

def modele_trois():
    for sous in ("chat/finale", "chat/meilleure", "lecture/finale"):
        if fait(f"{TROIS}/{sous}/config.json"):
            return f"{TROIS}/{sous}"

print("DATASET    =", DATASET); print("DATASET_UN =", DATASET_UN); print("DATASET_DEUX =", DATASET_DEUX)
print("DATASET_TROIS =", DATASET_TROIS); print("DATASET_QUATRE =", DATASET_QUATRE); print("TROIS       =", TROIS)
print("PUBLICS    =", PUBLICS); print("TRAVAIL    =", TRAVAIL)
for nom, ph in [("Lecture", "lecture"), ("Discussion", "chat")]:
    d = f"{TRAVAIL}/{ph}"
    print(f"{nom:11s}: " + ("FINI" if fait(f"{d}/finale") else "en cours (reprise)" if fait(f"{d}/derniere")
                             else "pas commencé"))

In [ ]:
# 0.c  Contrôle : chaque fichier a-t-il le nombre de lignes prévu ? (run trois : la Bible avait 9 000 lignes
#      sur Kaggle au lieu de 16 000, personne ne l'a vu). Sinon : arrêt tout de suite, pas 10 h plus tard.
import json
RACINES = {"dataset": DATASET, "dataset_un": DATASET_UN, "dataset_deux": DATASET_DEUX,
           "dataset_trois": DATASET_TROIS, "dataset_quatre": DATASET_QUATRE}
erreurs = []
for f, n in json.load(open(f"{DATASET_QUATRE}/attendu.json", encoding="utf-8")).items():
    racine, reste = f.split("/", 1)
    chemin = f"{RACINES[racine]}/{reste}"
    vu = sum(1 for l in open(chemin, encoding="utf-8") if l.strip()) if os.path.exists(chemin) else 0
    print(f"{'OK ' if vu == n else 'ERR'} {vu:7d} / {n:7d}  {f}")
    if vu != n:
        erreurs.append(f)
if erreurs:
    raise SystemExit("Dataset pas à jour : " + ", ".join(erreurs) + ". Refais une version du dataset modele-train.")

## Étape 1 : données publiques
**Quatre** fichiers Wikipédia (001 à 004, 20 M de mots chacun), pour moins répéter les mêmes textes.

In [ ]:
os.chdir(CODE)
if not glob.glob(f"{PUBLICS}/lecture_fr/wikipedia_fr_004.txt"):
    !python -u collecte/telecharger_publics.py --sortie {PUBLICS} --mots 80000000 2>&1 | grep -v "examples/s" | tee {J}/publics.txt
!ls {PUBLICS}/lecture_fr | head -3; ls {PUBLICS}/dialogues

## Étape 2 : lecture (continue la lecture du modèle trois)
2 époques (~2 h sur 2 x T4), le meilleur point est gardé. Si la session finit avant, pause propre et reprise.

In [ ]:
os.chdir(CODE)
if not PAUSE and not fait(f"{TRAVAIL}/lecture/finale"):
    !{LANCER} --config configs/quatre_lecture.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/lecture.txt
apres("lecture", f"{TRAVAIL}/lecture")

## Étape 3 : discussion (6 époques, ~2,5 h, le meilleur point est gardé)

In [ ]:
os.chdir(CODE)
if not PAUSE and not fait(f"{TRAVAIL}/chat/finale"):
    !{LANCER} --config configs/quatre_chat.yaml 2>&1 | grep --line-buffered -vE 'it/s\]|examples/s' | tee -a {J}/chat.txt
if not PAUSE:
    apres("chat", f"{TRAVAIL}/chat")

## Étape 4 : tests
**Exactement les mêmes tests** que le deux et le trois (`dialogues_test.jsonl` + FLORES devtest, jamais vus).
Une seule passe, **sans pénalité** (le run deux a montré que c'est mieux). Puis comparaison trois / quatre.
`+` = mieux, `-` = moins bien, `=` = pareil (écart de chrF).

In [ ]:
os.chdir(CODE)
if PAUSE:
    print("Tests sautés : l'entraînement n'est pas fini (version suivante).")
else:
    tests = "tests/dialogues_test.jsonl tests/flores.jsonl"
    for nom, modele in [("trois", modele_trois()), ("quatre", meilleur_modele())]:
        print("=====", nom, modele)
        !python -u evaluer.py --modele {modele} --tests {tests} --max 100 --montrer 3 \
            --sortie {J}/resultats_{nom}.jsonl 2>&1 | tee {J}/notes_{nom}.txt
    print("\n##### COMPARAISON trois / quatre (sans pénalité)")
    !python comparer_resultats.py {J}/resultats_trois.jsonl {J}/resultats_quatre.jsonl --noms trois quatre | tee {J}/comparaison.txt

## Étape 5 : essayer le modèle

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
appareil = "cuda" if torch.cuda.is_available() else "cpu"
dossier = meilleur_modele()
tok = AutoTokenizer.from_pretrained(dossier)
model = AutoModelForCausalLM.from_pretrained(dossier).to(appareil).eval()
historique = ""

def demander(question, garder=True):
    """Pose une question. garder=True : le modèle se souvient des tours d'avant."""
    global historique
    prompt = f"{historique}<|user|> {question} <|assistant|>"
    ids = tok(prompt, return_tensors="pt", return_token_type_ids=False).to(appareil)
    out = model.generate(**ids, max_new_tokens=150, do_sample=True, temperature=0.3, top_p=0.9,
                         repetition_penalty=1.05, eos_token_id=tok.eos_token_id, pad_token_id=tok.pad_token_id)
    reponse = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    if garder:
        historique = f"{prompt} {reponse}<|endoftext|>"
    return reponse

print(dossier)
for q in ["Bonjour", "Bwakeye.", "N'amaki?", "Où se trouve le Burundi ?", "Quels sont les pays voisins du Burundi ?",
          "Traduis en français : Umwana n'umukobwa.", "Que veut dire « Guhodoka » en kirundi ?",
          "Ingingo ya 5 y'Ibwirizwa shingiro rya 2018 ivuga iki?", "Que signifie « karangura » en grammaire kirundi ?",
          "Mbwira ingingo ya 26 y'Ibwirizwa shingiro rya 2018 uko yanditse."]:
    print(q, "->", demander(q, garder=False))

## Étape 6 : le `.zip` pour le backend
À mettre dans `litgpt/models/local/`, puis `python scripts/convert_local_model.py <dossier>`.

In [ ]:
if PAUSE:
    print("PAS FINI : relance 'Exécuter tout', ça reprend.")
else:
    dossier_modele = meilleur_modele()
    !cd "{dossier_modele}" && zip -qr /content/modele_quatre_fr_rn.zip . -x "etat_entrainement.pt" && cp /content/modele_quatre_fr_rn.zip {TRAVAIL}/
    from google.colab import files
    files.download("/content/modele_quatre_fr_rn.zip")